In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [ ]:
# --- LOAD DATA (doar ingrediente + denumire) ---
import pandas as pd
ingredients_path = '../output/ingredients_nutrional_profiles/ingredients_nutrional_profiles.parquet/part-00000-33e03f78-d7e8-41fa-a09f-4a8c4605dc35-c000.snappy.parquet'
df_ing = pd.read_parquet(ingredients_path)
foods = df_ing[['fdc_id', 'description', 'all_ingredients']].copy()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/13 17:23:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/13 17:23:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [ ]:
# --- CREATE TEXT FOR EMBEDDING ---
def join_ingredients(ing_list):
    if not isinstance(ing_list, list): return ''
    return ', '.join([str(i) for i in ing_list])
foods['text_for_embedding'] = foods['description'].fillna('') + '. Ingredients: ' + foods['all_ingredients'].apply(join_ingredients)

In [4]:
# --- LOAD EMBEDDING MODEL ---
model = SentenceTransformer('all-MiniLM-L6-v2')  # Rapid și bun pentru semantic search

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
# --- COMPUTE EMBEDDINGS ---
embeddings = model.encode(foods['text_for_embedding'].tolist(), show_progress_bar=True, convert_to_numpy=True)

Batches:   0%|          | 0/147 [00:00<?, ?it/s]

In [6]:
# --- FUNCTION: RECOMMEND SIMILAR PRODUCTS ---
def recommend_similar_products(query_text, top_n=5):
    query_emb = model.encode([query_text], convert_to_numpy=True)[0]
    scores = util.cos_sim(query_emb, embeddings)[0].cpu().numpy() if hasattr(util.cos_sim(query_emb, embeddings)[0], 'cpu') else util.cos_sim(query_emb, embeddings)[0].numpy()
    top_idx = np.argsort(scores)[::-1][:top_n]
    return foods.iloc[top_idx][['fdc_id', 'description', 'all_ingredients']], scores[top_idx]

## Exemplu de utilizare: Recomandă produse similare cu un aliment dat
Modifică textul de query pentru a găsi produse similare semantic.

In [8]:
# Exemplu: Recomandă produse similare cu un produs dat
query = 'PEPPERIDGE FARM BREAD GARLIC. Ingredients: enriched wheat flour, water, margarine, mozzarella cheese, yeast, sugar, garlic powder, parsley flakes'
results, sim_scores = recommend_similar_products(query, top_n=20)
results['similarity'] = sim_scores
display(results)

,fdc_id,description,all_ingredients,similarity
0,1105918,PEPPERIDGE FARM BREAD GARLIC,"[made from: enriched wheat flour (flour, niac...",0.917576
128,356055,PEPPERIDGE FARM BREAD FIVE CHEESE GARLIC,"[made from: enriched wheat flour (flour, niac...",0.842957
4089,2491582,"Pepperidge Farm Goldfish Cheddar Crackers, Bak...","[made with smiles and whole wheat flour, enri...",0.763401
3576,2231443,Pepperidge Farm Milano Cinnamon Chocolate Cook...,"[made from: enriched wheat flour (flour, niac...",0.754622
4555,2688641,Pepperidge Farm Nantucket Crispy Double Dark C...,"[made from: semi sweet chocolate (sugar, choc...",0.741796
3532,2197049,"Pepperidge Farm Goldfish Cheddar Crackers, 6.6...",[made with smiles and enriched wheat flour (fl...,0.726304
4580,2697750,"Pepperidge Farm Butter Hot Dog Buns, Top Slice...","[made from: enriched wheat flour (flour, niac...",0.723265
4091,2492601,Pepperidge Farm Goldfish Whole Grain Snack Cra...,"[made with smiles and whole wheat flour, enri...",0.720893
4456,2650506,Pepperidge Farm Captiva Dark Chocolate Cookies...,"[made from: enriched wheat flour (flour, niac...",0.705565
4581,2697784,Pepperidge Farm Milano London Fog Earl Grey Fl...,"[made from: enriched wheat flour (flour, niac...",0.704489
